<a href="https://colab.research.google.com/github/shruthilakshmi008-BA/ba-automation-suite/blob/main/06-Enterprise-O2C-Pipeline/06_Enterprise_O2C_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U google-genai pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [ ]:
import os
import re
import json
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import userdata

# Get Gemini API key securely from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets (🔑) and enable notebook access."
    )

from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.
Gemini client initialized successfully.


In [ ]:
STATUS_MAP = {
    "ok": "APPROVED",
    "approved": "APPROVED",
    "verified": "APPROVED",
    "pending": "PENDING_REVIEW",
    "review required": "PENDING_REVIEW",
    "flagged": "PENDING_REVIEW",
    "failed": "REJECTED",
    "rejected": "REJECTED"
}

INVOICE_PATTERNS = {
    "invoice_number": r"Invoice\s*#?:?\s*([A-Za-z0-9\-]+)",
    "vendor": r"Vendor:\s*([^\-\n]+)",
    "total": r"Total:\s*\$?([\d,]+\.\d{2})"
}

GEMINI_MODEL = "gemini-3.6-flash"

gemini_stats = {
    "calls": 0,
    "successes": 0,
    "errors": 0,
    "rate_limit_errors": 0
}

print("Configuration loaded.")

Configuration loaded.


In [ ]:
def call_gemini(prompt: str):
    """
    Central Gemini wrapper.
    Tracks successful calls, errors, and rate-limit errors.
    """
    gemini_stats["calls"] += 1

    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )

        gemini_stats["successes"] += 1
        return response.text.strip()

    except Exception as e:
        error_text = str(e).lower()

        if (
            "429" in error_text
            or "rate limit" in error_text
            or "quota" in error_text
            or "resource exhausted" in error_text
        ):
            gemini_stats["rate_limit_errors"] += 1

        gemini_stats["errors"] += 1

        print(f"[Gemini Error] {e}")
        return None


def normalize_status(raw_status: str):
    """
    Layer 1: Rule-based status mapping.
    Layer 2: Gemini fallback for unknown statuses.
    """
    cleaned = str(raw_status).strip().lower()

    if cleaned in STATUS_MAP:
        return STATUS_MAP[cleaned], "Rule Engine"

    prompt = f"""
You are an enterprise order-processing classification system.

Classify the following raw order status into exactly ONE of:

APPROVED
PENDING_REVIEW
REJECTED

Raw status:
{raw_status}

Return ONLY the category name.
"""

    result = call_gemini(prompt)

    if result:
        result = result.strip().upper()

        if result in ["APPROVED", "PENDING_REVIEW", "REJECTED"]:
            return result, "Gemini"

    return "PENDING_REVIEW", "Fallback Review"

In [ ]:
def parse_invoice_text(raw_text: str):
    """
    Layer 1: Regex extraction.
    Missing fields remain NEEDS_REVIEW.
    """
    extracted = {}

    for field, pattern in INVOICE_PATTERNS.items():
        match = re.search(pattern, raw_text, re.IGNORECASE)

        if match:
            extracted[field] = match.group(1).strip()
        else:
            extracted[field] = "NEEDS_REVIEW"

    return extracted


def extract_missing_invoice_metadata(raw_text: str, missing_fields: list):
    """
    Layer 2: Gemini fallback for missing invoice metadata.
    Only requested missing fields are sent for extraction.
    """

    if not missing_fields:
        return {}, "Not Required"

    prompt = f"""
You are an expert invoice document extraction system.

Extract ONLY these missing fields from the invoice text:

{missing_fields}

Invoice text:
{raw_text}

Return ONLY valid JSON.

Allowed fields:
- invoice_number: string
- vendor: string
- total: numeric string without currency symbols

If a field cannot be confidently identified, return null.

Example:
{{
    "invoice_number": "INV-1001",
    "vendor": "ABC Corporation",
    "total": "6400.00"
}}
"""

    result = call_gemini(prompt)

    if not result:
        return {
            field: "NEEDS_REVIEW"
            for field in missing_fields
        }, "Gemini Failed"

    try:
        if result.startswith("```"):
            result = re.sub(
                r"^```[a-zA-Z]*\n|\n```$",
                "",
                result,
                flags=re.MULTILINE
            )

        parsed = json.loads(result)

        output = {}

        for field in missing_fields:
            value = parsed.get(field)

            if value is None or str(value).strip() == "":
                output[field] = "NEEDS_REVIEW"
            else:
                output[field] = str(value).strip()

        return output, "Gemini"

    except Exception as e:
        print(f"[Metadata Parse Warning] {e}")

        return {
            field: "NEEDS_REVIEW"
            for field in missing_fields
        }, "Gemini Invalid Response"

In [ ]:
def generate_sdlc_artifacts():

    brd_content = """# Business Requirements Document (BRD)

## Project
Enterprise Order-to-Cash (O2C) Automated Exception Management System

---

## BR-001 — Automated Status Normalization

The system shall normalize incoming order status values into standardized
categories: APPROVED, PENDING_REVIEW, or REJECTED.

## BR-002 — Invoice Metadata Extraction

The system shall extract invoice number, vendor, and invoice total from
unstructured invoice text using deterministic rules with Gemini fallback.

## BR-003 — High-Value Compliance Control

The system shall identify transactions with invoice totals greater than
or equal to $5,000 for additional review.

## BR-004 — Exception Management

The system shall capture all applicable exception reasons for each order.

## BR-005 — Human Review Control

The system shall identify records requiring manual verification.

## BR-006 — Agile Backlog Generation

The system shall generate user stories for order exceptions.

## BR-007 — Operational Analytics

The system shall generate operational metrics and an executive briefing.

## BR-008 — AI Governance Tracking

The system shall track Gemini calls, successes, errors, and rate-limit events.
"""

    frd_content = """# Functional Requirements Document (FRD)

## FR-001 — Status Normalization

Maps known statuses using STATUS_MAP and uses Gemini for unmapped statuses.

**Traceability:** BR-001

## FR-002 — Regex Invoice Extraction

Extracts invoice_number, vendor, and total using deterministic patterns.

**Traceability:** BR-002

## FR-003 — Gemini Metadata Fallback

When regex extraction fails, Gemini attempts extraction of only the missing fields.

**Traceability:** BR-002

## FR-004 — Missing Total Protection

Missing invoice totals shall remain NEEDS_REVIEW and shall never be converted
to zero.

**Traceability:** BR-002, BR-005

## FR-005 — Multiple Exception Reasons

The system shall preserve all applicable exception reasons for each transaction.

**Traceability:** BR-004

## FR-006 — High-Value Threshold

Transactions with numeric totals >= $5,000 shall receive an
Exceeds $5k Approval Threshold exception.

**Traceability:** BR-003

## FR-007 — Human Review Flag

The system shall set needs_human_review = Yes whenever one or more
exceptions require manual verification.

**Traceability:** BR-005

## FR-008 — Agile Story Generation

The system shall generate a user story for each exception record.

**Traceability:** BR-006

## FR-009 — Executive Analytics

The system shall calculate pipeline metrics and generate an executive briefing.

**Traceability:** BR-007

## FR-010 — Gemini Monitoring

The system shall record Gemini calls, successes, errors, and rate-limit errors.

**Traceability:** BR-008
"""

    rtm_content = """# Requirements Traceability Matrix (RTM)

| Business Requirement | Functional Requirement | System Function | UAT Test |
|---|---|---|---|
| BR-001 | FR-001 | normalize_status() | TC-UAT-001 |
| BR-002 | FR-002 | parse_invoice_text() | TC-UAT-002 |
| BR-002 | FR-003 | extract_missing_invoice_metadata() | TC-UAT-003 |
| BR-002 | FR-004 | Missing total protection | TC-UAT-004 |
| BR-004 | FR-005 | Multiple exception reasons | TC-UAT-005 |
| BR-003 | FR-006 | $5k threshold engine | TC-UAT-006 |
| BR-005 | FR-007 | needs_human_review | TC-UAT-007 |
| BR-006 | FR-008 | generate_user_story() | TC-UAT-008 |
| BR-007 | FR-009 | Executive analytics | TC-UAT-009 |
| BR-008 | FR-010 | Gemini monitoring | TC-UAT-010 |
"""

    uat_content = """# User Acceptance Testing (UAT) Plan

## TC-UAT-001 — Status Normalization

**BR:** BR-001
**FR:** FR-001

Given an order has raw status "approved"

When the pipeline processes the record

Then canonical status shall be APPROVED.

---

## TC-UAT-002 — Regex Invoice Extraction

**BR:** BR-002
**FR:** FR-002

Given invoice text contains standard invoice fields

When regex extraction runs

Then available metadata shall be extracted.

---

## TC-UAT-003 — Gemini Metadata Fallback

**BR:** BR-002
**FR:** FR-003

Given an invoice field is missing from regex extraction

When Gemini fallback is executed

Then Gemini shall attempt extraction of the missing field.

---

## TC-UAT-004 — Missing Total Protection

**BR:** BR-002, BR-005
**FR:** FR-004

Given an invoice contains no identifiable total

When processing completes

Then total shall remain NEEDS_REVIEW and shall NOT become 0.

---

## TC-UAT-005 — Multiple Exceptions

**BR:** BR-004
**FR:** FR-005

Given an order has multiple problems

When exception evaluation executes

Then all applicable exception reasons shall be retained.

---

## TC-UAT-006 — High Value Transaction

**BR:** BR-003
**FR:** FR-006

Given invoice total is $6,400

When threshold evaluation executes

Then the order shall be flagged with Exceeds $5k Approval Threshold.

---

## TC-UAT-007 — Human Review Flag

**BR:** BR-005
**FR:** FR-007

Given an order contains an unresolved exception

When processing completes

Then needs_human_review shall equal Yes.

---

## TC-UAT-008 — User Story

**BR:** BR-006
**FR:** FR-008

Given an exception exists

When story generation executes

Then an Agile user story shall be generated.

---

## TC-UAT-009 — Analytics

**BR:** BR-007
**FR:** FR-009

Given processed O2C records exist

When analytics execute

Then pipeline metrics and executive briefing shall be produced.

---

## TC-UAT-010 — Gemini Monitoring

**BR:** BR-008
**FR:** FR-010

Given Gemini calls occur during processing

When the pipeline completes

Then Gemini calls, successes, errors, and rate-limit events shall be reported.
"""

    with open("01_BRD_Business_Requirements_Document.md", "w") as f:
        f.write(brd_content)

    with open("02_FRD_Functional_Requirements_Document.md", "w") as f:
        f.write(frd_content)

    with open("03_Traceability_Matrix_RTM.md", "w") as f:
        f.write(rtm_content)

    with open("04_UAT_Acceptance_Test_Plan.md", "w") as f:
        f.write(uat_content)

    print("SDLC artifacts generated successfully.")

In [ ]:
def generate_user_story(order_row: dict):

    prompt = f"""
You are a Technical Business Analyst.

Create a professional Agile user story for this O2C exception.

Order ID: {order_row['order_id']}
Vendor: {order_row['vendor']}
Invoice Total: {order_row['total']}
Status: {order_row['canonical_status']}
Exception Reasons: {order_row['exception_reasons']}

Return:

User Story:
As a [role], I want to [capability], so that [business value].

Acceptance Criteria:
- Given...
- When...
- Then...
"""

    result = call_gemini(prompt)

    if result:
        return result

    return (
        f"**User Story:** As a Finance Operations Analyst, I want to review "
        f"flagged Order {order_row['order_id']} so that transaction risks "
        f"are controlled.\n\n"
        f"**Acceptance Criteria:**\n"
        f"- Given an exception exists\n"
        f"- When the order enters the review process\n"
        f"- Then the exception shall be manually verified."
    )


def generate_executive_brief(metrics: dict):

    prompt = f"""
You are a Lead Business Analyst.

Analyze these O2C operational metrics:

{json.dumps(metrics, indent=2)}

Provide exactly three sections:

**Executive Takeaway:**
Overall operational summary.

**Compliance & Exception Bottlenecks:**
Key risks and exception patterns.

**Next Actions:**
Concrete business improvement recommendations.
"""

    result = call_gemini(prompt)

    if result:
        return result

    return (
        f"**Executive Takeaway:** The O2C pipeline processed "
        f"{metrics['total_orders']} orders with an approval rate of "
        f"{metrics['approval_rate_pct']}%.\n\n"

        f"**Compliance & Exception Bottlenecks:** "
        f"{metrics['exception_count']} orders require exception handling "
        f"or human review.\n\n"

        f"**Next Actions:** Prioritize unresolved invoice metadata, "
        f"high-value transactions, and recurring status exceptions."
    )

In [ ]:
def render_pipeline_dashboard(df: pd.DataFrame, output_path: str):

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # Chart 1: Status Distribution
    status_counts = df["canonical_status"].value_counts()

    axes[0].bar(
        status_counts.index,
        status_counts.values,
        color=["#5cb85c", "#f0ad4e", "#d9534f"]
    )

    axes[0].set_title(
        "O2C Order Volume by Canonical Status",
        fontsize=11,
        fontweight="bold"
    )

    axes[0].set_ylabel("Order Count")

    # Chart 2: Order Values
    colors = [
        "#d9534f" if flag != "NONE" else "#2b5c8f"
        for flag in df["exception_reasons"]
    ]

    axes[1].bar(
        df["order_id"].astype(str),
        df["numeric_total"],
        color=colors
    )

    axes[1].axhline(
        5000,
        color="red",
        linestyle="--",
        label="High-Value Threshold ($5,000)"
    )

    axes[1].set_title(
        "Order Value & Exception Flags",
        fontsize=11,
        fontweight="bold"
    )

    axes[1].set_ylabel("Total Amount ($)")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

    print(f"Dashboard saved to: {output_path}")

In [ ]:
def run_o2c_pipeline(
    input_csv,
    output_csv="o2c_processed_orders.csv",
    stories_path="05_User_Stories_Backlog.md"
):

    if not os.path.exists(input_csv):
        print(f"Error: {input_csv} not found.")
        return

    # Reset Gemini tracking for this run
    for key in gemini_stats:
        gemini_stats[key] = 0

    # Generate SDLC documentation
    generate_sdlc_artifacts()

    # Read input
    df = pd.read_csv(input_csv)

    processed_rows = []

    user_stories = [
        "# User Stories Backlog (Agile Exception Items)\n\n"
    ]

    for idx, row in df.iterrows():

        order_id = row["order_id"]
        raw_status = row["raw_status"]
        raw_invoice_text = row["raw_invoice_text"]

        # ------------------------------------------------
        # Step 1 — Status Normalization
        # ------------------------------------------------
        canonical_status, status_method = normalize_status(raw_status)

        # ------------------------------------------------
        # Step 2 — Regex Invoice Extraction
        # ------------------------------------------------
        invoice_meta = parse_invoice_text(raw_invoice_text)

        missing_fields = [
            field
            for field, value in invoice_meta.items()
            if value == "NEEDS_REVIEW"
        ]

        metadata_method = "Regex"

        # ------------------------------------------------
        # Step 3 — Gemini Metadata Fallback
        # ------------------------------------------------
        if missing_fields:

            ai_metadata, ai_method = extract_missing_invoice_metadata(
                raw_invoice_text,
                missing_fields
            )

            for field, value in ai_metadata.items():
                if value != "NEEDS_REVIEW":
                    invoice_meta[field] = value

            metadata_method = f"Regex + {ai_method}"

        # ------------------------------------------------
        # Step 4 — Preserve Missing Total
        # ------------------------------------------------
        total_value = invoice_meta.get("total", "NEEDS_REVIEW")

        if total_value == "NEEDS_REVIEW":
            numeric_total = None
        else:
            try:
                numeric_total = float(
                    str(total_value).replace(",", "").replace("$", "")
                )
            except:
                numeric_total = None
                invoice_meta["total"] = "NEEDS_REVIEW"

        # ------------------------------------------------
        # Step 5 — Multiple Exception Reasons
        # ------------------------------------------------
        exception_reasons = []

        if canonical_status == "PENDING_REVIEW":
            exception_reasons.append("Unresolved Raw Status")

        if canonical_status == "REJECTED":
            exception_reasons.append("System Rejection")

        if invoice_meta["invoice_number"] == "NEEDS_REVIEW":
            exception_reasons.append("Missing Invoice Number")

        if invoice_meta["vendor"] == "NEEDS_REVIEW":
            exception_reasons.append("Missing Vendor")

        if invoice_meta["total"] == "NEEDS_REVIEW":
            exception_reasons.append("Missing Invoice Total")

        # IMPORTANT:
        # Only evaluate $5k when a real numeric total exists.
        if numeric_total is not None and numeric_total >= 5000:
            exception_reasons.append(
                "Exceeds $5k Approval Threshold"
            )
            canonical_status = "PENDING_REVIEW"

        # Remove duplicate reasons
        exception_reasons = list(dict.fromkeys(exception_reasons))

        if not exception_reasons:
            exception_reason_text = "NONE"
            needs_human_review = "No"
        else:
            exception_reason_text = "; ".join(exception_reasons)
            needs_human_review = "Yes"

        # ------------------------------------------------
        # Step 6 — Build Processed Record
        # ------------------------------------------------
        record = {
            "order_id": order_id,
            "vendor": invoice_meta["vendor"],
            "invoice_number": invoice_meta["invoice_number"],
            "total": invoice_meta["total"],
            "numeric_total": numeric_total,
            "raw_status": raw_status,
            "canonical_status": canonical_status,
            "exception_reasons": exception_reason_text,
            "needs_human_review": needs_human_review,
            "status_extraction_method": status_method,
            "metadata_extraction_method": metadata_method
        }

        processed_rows.append(record)

        # ------------------------------------------------
        # Step 7 — Agile User Story
        # ------------------------------------------------
        if needs_human_review == "Yes":

            print(
                f"Generating story for {order_id}: "
                f"{exception_reason_text}"
            )

            story_text = generate_user_story(record)

            user_stories.append(
                f"### Story: Exception - Order {order_id}\n"
                f"{story_text}\n\n"
                f"---\n\n"
            )

    # ------------------------------------------------
    # Step 8 — Save Processed CSV
    # ------------------------------------------------
    processed_df = pd.DataFrame(processed_rows)

    processed_df.to_csv(
        output_csv,
        index=False
    )

    # ------------------------------------------------
    # Step 9 — Save User Stories
    # ------------------------------------------------
    with open(stories_path, "w") as f:
        f.writelines(user_stories)

    # ------------------------------------------------
    # Step 10 — Metrics
    # ------------------------------------------------
    total_orders = len(processed_df)

    approved_count = len(
        processed_df[
            processed_df["canonical_status"] == "APPROVED"
        ]
    )

    exception_count = len(
        processed_df[
            processed_df["needs_human_review"] == "Yes"
        ]
    )

    pipeline_metrics = {
        "total_orders": total_orders,
        "approved_orders": approved_count,
        "exception_count": exception_count,
        "approval_rate_pct": round(
            (approved_count / total_orders) * 100, 1
        ) if total_orders else 0,
        "total_pipeline_value": round(
            processed_df["numeric_total"].dropna().sum(), 2
        ),
        "gemini_calls": gemini_stats["calls"],
        "gemini_successes": gemini_stats["successes"],
        "gemini_errors": gemini_stats["errors"],
        "gemini_rate_limit_errors": gemini_stats["rate_limit_errors"]
    }

    # ------------------------------------------------
    # Step 11 — Dashboard
    # ------------------------------------------------
    render_pipeline_dashboard(
        processed_df,
        "o2c_pipeline_dashboard.png"
    )

    # ------------------------------------------------
    # Step 12 — Executive Brief
    # ------------------------------------------------
    executive_brief = generate_executive_brief(
        pipeline_metrics
    )

    print("\n================ O2C PIPELINE EXECUTIVE BRIEFING ================\n")

    print(executive_brief)

    print("\n==================================================================")

    print(f"\nProcessed Dataset : {output_csv}")
    print(f"User Stories     : {stories_path}")
    print("Dashboard        : o2c_pipeline_dashboard.png")

    print("\n================ GEMINI USAGE =================")
    print(f"Gemini Calls       : {gemini_stats['calls']}")
    print(f"Successful Calls   : {gemini_stats['successes']}")
    print(f"Errors             : {gemini_stats['errors']}")
    print(f"Rate Limit Errors  : {gemini_stats['rate_limit_errors']}")
    print("===============================================\n")

    return processed_df, pipeline_metrics, executive_brief

In [ ]:
from google.colab import files

print("Upload your O2C input CSV file.")

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

print(f"\nInput file selected: {input_file}")

processed_df, metrics, executive_brief = run_o2c_pipeline(
    input_csv=input_file,
    output_csv="o2c_processed_orders.csv",
    stories_path="05_User_Stories_Backlog.md"
)


files.download("o2c_processed_orders.csv")
files.download("05_User_Stories_Backlog.md")
files.download("o2c_pipeline_dashboard.png")
files.download("03_Traceability_Matrix_RTM.md")
files.download("04_UAT_Acceptance_Test_Plan.md")
files.download("01_BRD_Business_Requirements_Document.md")
files.download("02_FRD_Functional_Requirements_Document.md")

Upload your O2C input CSV file.


Saving ingestion_file.csv to ingestion_file (1).csv

Input file selected: ingestion_file (1).csv
SDLC artifacts generated successfully.
Generating story for ORD-902: Unresolved Raw Status; Exceeds $5k Approval Threshold
Generating story for ORD-904: Unresolved Raw Status; Exceeds $5k Approval Threshold
Dashboard saved to: o2c_pipeline_dashboard.png

================ O2C PIPELINE EXECUTIVE BRIEFING ================

**Executive Takeaway:**
The O2C pipeline processed 5 orders representing a total pipeline value of $20,100. While automated AI processing via Gemini exhibited perfect operational reliability (100% success rate across 3 calls with zero system or rate-limit errors), overall business throughput is constrained by a sub-optimal order approval rate of 60.0%. Two out of five orders were diverted to exception workflows, creating friction in revenue conversion despite a stable underlying technical infrastructure.

**Compliance & Exception Bottlenecks:**
* **High Operational Exception

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>